# Test DFA-2-RegExp

In [ ]:
// Checks remain active in both F# Interactive and the Jupyter kernel.
let checkNotebook condition =
    if not condition then failwith "Notebook check failed"


In [ ]:
#load "AutomataTypes.fsx"
#load "RegExpParser.fsx"
#load "NfaToDfa.fsx"
#load "RegExpToNfa.fsx"
#load "DfaToRegExp.fsx"
#load "Rewrite.fsx"
#load "FsmToDot.fsx"

In [ ]:
open AutomataTypes
open RegExpParser
open NfaToDfa
open RegExpToNfa
open DfaToRegExp
open Rewrite
open FsmToDot

In [ ]:
let delta = Map.ofList [ (1, 'a'), 2; (2, 'b'), 3; (3, 'a'), 2 ]
let automaton : Dfa<int> =
    { States = set [1;2;3]; Alphabet = set ['a';'b']; Transitions = delta
      Start = 1; Accepting = set [3] }
let originalDot, _ = dfa2dot automaton
originalDot

In [ ]:
Microsoft.DotNet.Interactive.Kernel.display(Microsoft.DotNet.Interactive.Kernel.HTML(renderSvg originalDot), "text/html") |> ignore

In [ ]:
let r = dfa2regexp automaton
r

As this regular expression is nearly unreadable,  The notebook `Rewrite.ipynb` contains the definition of the function `simplify` that can be used to simplify this expression.

In [ ]:
let simplified = simplify r
simplified

The formatter is defined in the conversion notebook and retains grouping. This automaton accepts `ab(ab)*`. Its transition map is partial, so a missing transition rejects a word.

In [ ]:
printfn "%s" (regexp2string simplified)

In [ ]:
let roundTrip = regexp2string simplified |> parse |> regexp2nfa automaton.Alphabet |> nfa2dfa
let rec words length =
    if length = 0 then [""]
    else [ for prefix in words (length - 1) do for c in automaton.Alphabet do yield prefix + string c ]
for length in 0 .. 7 do
    for word in words length do
        checkNotebook (acceptsDfa automaton word = acceptsDfa roundTrip word)
checkNotebook (acceptsDfa roundTrip "ab")
checkNotebook (acceptsDfa roundTrip "abab")
checkNotebook (not (acceptsDfa roundTrip "aba"))